## Try to merge flight data with weather

In [1]:
%pwd

'/Users/nicholasstanfield/Desktop/flight-delay/notebooks'

In [2]:
import pandas as pd

df = pd.read_csv("../data/processed/flight_data_2025.csv")
df.head()

,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,ORIGIN,DEST,CRS_DEP_TIME,CRS_ARR_TIME,CRS_ELAPSED_TIME,DISTANCE,AIRLINE,ROUTE,DELAY
0,1,1,1,3,AUS,ORD,545,830,165.0,977.0,American Airlines Inc.,AUS-ORD,0
1,1,1,7,2,PBI,DFW,620,842,202.0,1102.0,American Airlines Inc.,PBI-DFW,0
2,1,1,26,7,LAS,DEN,920,1219,119.0,628.0,United Air Lines Inc.,LAS-DEN,0
3,1,1,17,5,ICT,ATL,1750,2105,135.0,782.0,Delta Air Lines Inc.,ICT-ATL,0
4,1,1,14,2,CHS,EWR,1930,2129,119.0,628.0,United Air Lines Inc.,CHS-EWR,0


## Pipeline Steps

1. Create a list of all the airline codes
2. Read in the airport coordinates (with the right columns)
3. Merge the codes and coordinates
4. Add the timezone
5. Merge on ORIGIN codes
6. Merge on DEPT codes
7. Create SCHEDULED DEPT and ARR datetime columns
8. Convert these to UTC (maybe optional if you can use the timezone in the weather API
9. Floor/round to get in date hour format e.g. 2025-01-01 14:00:00+00:00
10. Load in weather data
11. Create a dep_weather and arrival_weather
12. Join each onto main dataframe
13. Drop unnecessary columns
14. Done!

In [3]:
codes = list(set(list(df["ORIGIN"].unique()) + list(df["DEST"].unique()))) # grab all the airports in the dataset
codes = pd.DataFrame(codes)
codes.columns = ["code"]
codes

,code
0,EVV
1,CDV
2,HDN
3,JAC
4,HPN
...,...
346,DAL
347,EKO
348,JFK
349,ISP


In [5]:
# the data below is from the following website: https://ourairports.com/help/data-dictionary.html
# it is the airports.csv

In [4]:
coords = pd.read_csv("../data/raw/airport_coordinates.csv") 
coords = coords[["iata_code", "latitude_deg", "longitude_deg"]]
coords

,iata_code,latitude_deg,longitude_deg
0,NaN,40.070985,-74.933689
1,NaN,38.704022,-101.473911
2,NaN,59.947733,-151.692524
3,NaN,34.864799,-86.770302
4,NaN,59.093287,-156.456699
...,...,...,...
85931,NaN,41.784354,123.496308
85932,NaN,51.894444,1.482500
85933,NaN,-11.584278,47.296389
85934,NaN,32.110587,-97.356312


In [5]:
airports = pd.merge(codes, coords, left_on="code", right_on="iata_code", how="inner")
airports

,code,iata_code,latitude_deg,longitude_deg
0,EVV,EVV,38.036999,-87.532402
1,CDV,CDV,60.491798,-145.477997
2,HDN,HDN,40.481201,-107.218002
3,JAC,JAC,43.607300,-110.737999
4,HPN,HPN,41.067001,-73.707603
...,...,...,...,...
345,DAL,DAL,32.844776,-96.847653
346,EKO,EKO,40.824902,-115.792000
347,JFK,JFK,40.639447,-73.779317
348,ISP,ISP,40.796324,-73.101703


In [6]:
airports = airports.drop("code", axis=1)
airports.head()

,iata_code,latitude_deg,longitude_deg
0,EVV,38.036999,-87.532402
1,CDV,60.491798,-145.477997
2,HDN,40.481201,-107.218002
3,JAC,43.607300,-110.737999
4,HPN,41.067001,-73.707603


In [8]:
# Now we have the airport coord look up table we need to get the weather data
# For this we need the coords + some time constraint
# Most natural grain would be the hour (minute is too much data, daily too imprecise)
# So we will have DEPT_LAT, DEPT_LON, DEPT_HOUR and ARR_LAT, ARR_LON, ARR_HOUR
# Then merge some weather data (to be found) on each row in two ways
# 1. merge on the DEPT location and hour 2. merge on ARR location and hour
# So need to organize the time in original df properly

In [9]:
# important!
# need to account for timezones and find out what CRS_DEP_TIME and CRS_ARR_TIME timezones are
# seems like they are local time so will need to add timezones to the airport coords to account for time

In [7]:
from timezonefinder import TimezoneFinder
tf = TimezoneFinder()

airports["timezone"] = airports.apply(lambda row: tf.timezone_at(lat=row["latitude_deg"], lng=row["longitude_deg"]), axis=1)
airports.head() 

,iata_code,latitude_deg,longitude_deg,timezone
0,EVV,38.036999,-87.532402,America/Chicago
1,CDV,60.491798,-145.477997,America/Anchorage
2,HDN,40.481201,-107.218002,America/Denver
3,JAC,43.607300,-110.737999,America/Denver
4,HPN,41.067001,-73.707603,America/New_York


In [11]:
# now we can merge this new info into main df

In [8]:
df = pd.read_csv("../data/processed/flight_data_2025.csv")

In [9]:
len(df)

360000

In [ ]:
# less than 360_000 so we have some missing values

In [10]:
test = df.merge(airports, left_on="ORIGIN", right_on="iata_code", how="left")
missing = test[test["timezone"].isna()]
missing["ORIGIN"].value_counts()

ORIGIN
PBI    1573
Name: count, dtype: int64

In [15]:
# the airport name has changed since 2025 need to add the original into the airport table 

In [11]:
new_airport = {
    "iata_code": "PBI",
    "latitude_deg": 26.6832,
    "longitude_deg": -80.0956,
    "timezone": "America/New_York"
}

airports.loc[len(airports)] = new_airport

In [65]:
airports.tail(5)

,iata_code,latitude_deg,longitude_deg,timezone
346,EKO,40.824902,-115.792000,America/Los_Angeles
347,JFK,40.639447,-73.779317,America/New_York
348,ISP,40.796324,-73.101703,America/New_York
349,HHH,32.224400,-80.697502,America/New_York
350,PBI,26.683200,-80.095600,America/New_York


In [13]:
airports[airports["iata_code"]=='PBI']

,iata_code,latitude_deg,longitude_deg,timezone
350,PBI,26.6832,-80.0956,America/New_York


In [14]:
airports.to_csv("../data/raw/airport_coordinates_20260819.csv",index=False) # use for the pipeline in src

In [15]:
len(df.merge(airports, left_on="ORIGIN", right_on="iata_code", how="inner")) # now it works

360000

In [16]:
df = df.merge(airports, left_on="ORIGIN", right_on="iata_code", how="inner")
df = df.rename(columns={"latitude_deg":"ORIGIN_LATITUDE","longitude_deg":"ORIGIN_LONGITUDE","timezone":"ORIGIN_TIMEZONE"})
df = df.drop('iata_code', axis=1)
df.head()

,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,ORIGIN,DEST,CRS_DEP_TIME,CRS_ARR_TIME,CRS_ELAPSED_TIME,DISTANCE,AIRLINE,ROUTE,DELAY,ORIGIN_LATITUDE,ORIGIN_LONGITUDE,ORIGIN_TIMEZONE
0,1,1,1,3,AUS,ORD,545,830,165.0,977.0,American Airlines Inc.,AUS-ORD,0,30.197535,-97.662015,America/Chicago
1,1,1,7,2,PBI,DFW,620,842,202.0,1102.0,American Airlines Inc.,PBI-DFW,0,26.683200,-80.095600,America/New_York
2,1,1,26,7,LAS,DEN,920,1219,119.0,628.0,United Air Lines Inc.,LAS-DEN,0,36.083361,-115.151817,America/Los_Angeles
3,1,1,17,5,ICT,ATL,1750,2105,135.0,782.0,Delta Air Lines Inc.,ICT-ATL,0,37.650314,-97.428583,America/Chicago
4,1,1,14,2,CHS,EWR,1930,2129,119.0,628.0,United Air Lines Inc.,CHS-EWR,0,32.896159,-80.038151,America/New_York


In [17]:
df = df.merge(airports, left_on="DEST", right_on="iata_code", how="inner")
df = df.rename(columns={"latitude_deg":"DEST_LATITUDE","longitude_deg":"DEST_LONGITUDE","timezone":"DEST_TIMEZONE"})
df = df.drop('iata_code', axis=1)

In [19]:
df.head(3)

,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,ORIGIN,DEST,CRS_DEP_TIME,CRS_ARR_TIME,CRS_ELAPSED_TIME,DISTANCE,AIRLINE,ROUTE,DELAY,ORIGIN_LATITUDE,ORIGIN_LONGITUDE,ORIGIN_TIMEZONE,DEST_LATITUDE,DEST_LONGITUDE,DEST_TIMEZONE,SCHEDULED_DEP_DATETIME
0,1,1,1,3,AUS,ORD,545,830,165.0,977.0,American Airlines Inc.,AUS-ORD,0,30.197535,-97.662015,America/Chicago,41.978600,-87.904800,America/Chicago,2025-01-01 05:45:00
1,1,1,7,2,PBI,DFW,620,842,202.0,1102.0,American Airlines Inc.,PBI-DFW,0,26.683200,-80.095600,America/New_York,32.896801,-97.038002,America/Chicago,2025-01-07 06:20:00
2,1,1,26,7,LAS,DEN,920,1219,119.0,628.0,United Air Lines Inc.,LAS-DEN,0,36.083361,-115.151817,America/Los_Angeles,39.860027,-104.673792,America/Denver,2025-01-26 09:20:00


In [18]:
df["SCHEDULED_DEP_DATETIME"] = pd.to_datetime({
    "year": 2025,
    "month": df["MONTH"],
    "day": df["DAY_OF_MONTH"],
    "hour": df["CRS_DEP_TIME"] // 100,
    "minute": df["CRS_DEP_TIME"] % 100,
})

In [20]:
df["SCHEDULED_DEP_DATETIME"]

0        2025-01-01 05:45:00
1        2025-01-07 06:20:00
2        2025-01-26 09:20:00
3        2025-01-17 17:50:00
4        2025-01-14 19:30:00
                 ...        
359995   2025-12-28 17:14:00
359996   2025-12-21 09:20:00
359997   2025-12-18 12:40:00
359998   2025-12-11 14:35:00
359999   2025-12-09 07:26:00
Name: SCHEDULED_DEP_DATETIME, Length: 360000, dtype: datetime64[ns]

In [21]:
df["SCHEDULED_DEP_UTC"] = [
    dt.tz_localize(tz).tz_convert("UTC")
    for dt, tz in zip(
        df["SCHEDULED_DEP_DATETIME"],
        df["ORIGIN_TIMEZONE"]
    )
]

In [22]:
df["SCHEDULED_DEP_UTC"]

0        2025-01-01 11:45:00+00:00
1        2025-01-07 11:20:00+00:00
2        2025-01-26 17:20:00+00:00
3        2025-01-17 23:50:00+00:00
4        2025-01-15 00:30:00+00:00
                    ...           
359995   2025-12-28 22:14:00+00:00
359996   2025-12-21 14:20:00+00:00
359997   2025-12-18 17:40:00+00:00
359998   2025-12-11 20:35:00+00:00
359999   2025-12-09 15:26:00+00:00
Name: SCHEDULED_DEP_UTC, Length: 360000, dtype: datetime64[ns, UTC]

In [23]:
df["SCHEDULED_ARR_UTC"] = (
    df["SCHEDULED_DEP_UTC"]
    + pd.to_timedelta(df["CRS_ELAPSED_TIME"], unit="m")
)

df["SCHEDULED_ARR_UTC"]

0        2025-01-01 14:30:00+00:00
1        2025-01-07 14:42:00+00:00
2        2025-01-26 19:19:00+00:00
3        2025-01-18 02:05:00+00:00
4        2025-01-15 02:29:00+00:00
                    ...           
359995   2025-12-29 02:30:00+00:00
359996   2025-12-21 17:05:00+00:00
359997   2025-12-18 20:45:00+00:00
359998   2025-12-11 22:25:00+00:00
359999   2025-12-09 16:38:00+00:00
Name: SCHEDULED_ARR_UTC, Length: 360000, dtype: datetime64[ns, UTC]

In [24]:
df["DEP_WEATHER_HOUR"] = (
    df["SCHEDULED_DEP_UTC"].dt.floor("h")
)

df["ARR_WEATHER_HOUR"] = (
    df["SCHEDULED_ARR_UTC"].dt.floor("h")
)

In [25]:
df.head(2)

,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,ORIGIN,DEST,CRS_DEP_TIME,CRS_ARR_TIME,CRS_ELAPSED_TIME,DISTANCE,...,ORIGIN_LONGITUDE,ORIGIN_TIMEZONE,DEST_LATITUDE,DEST_LONGITUDE,DEST_TIMEZONE,SCHEDULED_DEP_DATETIME,SCHEDULED_DEP_UTC,SCHEDULED_ARR_UTC,DEP_WEATHER_HOUR,ARR_WEATHER_HOUR
0,1,1,1,3,AUS,ORD,545,830,165.0,977.0,...,-97.662015,America/Chicago,41.978600,-87.904800,America/Chicago,2025-01-01 05:45:00,2025-01-01 11:45:00+00:00,2025-01-01 14:30:00+00:00,2025-01-01 11:00:00+00:00,2025-01-01 14:00:00+00:00
1,1,1,7,2,PBI,DFW,620,842,202.0,1102.0,...,-80.095600,America/New_York,32.896801,-97.038002,America/Chicago,2025-01-07 06:20:00,2025-01-07 11:20:00+00:00,2025-01-07 14:42:00+00:00,2025-01-07 11:00:00+00:00,2025-01-07 14:00:00+00:00


In [26]:
# at this point we have many intermediate columns
# could drop all but dep_weather_hour and arr_weather_hour except we will drop these after adding weather so wait 
# till weather is merged for the final drop

## OpenMeteo API test

In [36]:
import requests
import time
from pathlib import Path
from tqdm.auto import tqdm

OUTPUT_DIR = Path("../data/raw/weather")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

weather_variables = [
    "temperature_2m",
    "precipitation",
    "snowfall",
    "snow_depth",
    "visibility",
    "surface_pressure",
    "wind_speed_10m",
    "wind_gusts_10m",
] # could add more but limited by Open-Meteo API pricing

# need to batch download weather data due to rate limits
# 3 airports ~ 70 API calls
# Limits: 600 calls per min, 5_000 per hour, 10_000 per day
# Main constraint is the hourly limit
# Do 3 airports per min = 70 * 60 = 4200 calls per hour
# Need 350 airports = 350/3 * 70 ~ 8,190 calls in total
BATCH_SIZE = 3
SLEEP_SECONDS = 60


In [37]:
def get_weather_batch(batch):
    
    url = "https://historical-forecast-api.open-meteo.com/v1/forecast"

    params = {
        "latitude": batch["latitude_deg"].tolist(),
        "longitude": batch["longitude_deg"].tolist(),
        "start_date": "2025-01-01",
        "end_date": "2025-12-31",
        "hourly": ",".join(weather_variables),
        "timezone": "GMT", # default but to be explicit we are using UTC/GMT timezone
    }

    response = requests.get(
        url,
        params=params,
        timeout=120
    )

    response.raise_for_status()

    data = response.json()

    # Multiple locations returns a list.
    # This also makes the function work with a one-row batch
    if isinstance(data, dict):
        data = [data]

    weather_dfs = []

    for airport_data, (_, airport) in zip(
        data,
        batch.iterrows()
    ):

        airport_weather = pd.DataFrame(
            airport_data["hourly"]
        )

        airport_weather["iata_code"] = airport["iata_code"]

        weather_dfs.append(airport_weather)

    return pd.concat(
        weather_dfs,
        ignore_index=True
    )

In [41]:
for start in tqdm(
    range(0, len(airports), BATCH_SIZE) # loop through all airport codes (350 in batches of 3) = 117 batches
):

    batch = airports.iloc[
        start:start + BATCH_SIZE
    ] # e.g. if start = 3 batch = airports.iloc[3:6] gets ['JAC', 'HPN', 'DDC']

    airport_codes = batch["iata_code"].tolist()

    file_name = "_".join(airport_codes)
    file_path = OUTPUT_DIR / f"{file_name}.parquet" #use parquet to save space as these will be heavy files

    # Resume check so if the loop fails we can quickly restart
    if file_path.exists():
        print(f"Skipping already downloaded: {airport_codes}")
        continue

    try:
        weather_batch = get_weather_batch(batch) #returns a dataframe of three aiports hourly data = 3*365*24 = 26,280 rows

        weather_batch["time"] = pd.to_datetime(
            weather_batch["time"],
            utc=True
        )

        weather_batch.to_parquet(
            file_path,
            index=False
        )

        print(f"Saved: {airport_codes}")

    except Exception as e:
        print(f"Failed {airport_codes}: {e}")

    time.sleep(SLEEP_SECONDS)

  0%|          | 0/117 [00:00<?, ?it/s]

Saved: ['EVV', 'CDV', 'HDN']
Saved: ['JAC', 'HPN', 'DDC']
Saved: ['MCO', 'TOL', 'SYR']
Saved: ['LRD', 'SWF', 'PLN']
Saved: ['MFR', 'TLH', 'FLL']
Saved: ['DTW', 'ELP', 'ROC']
Saved: ['CLT', 'GST', 'BLV']
Saved: ['SAT', 'JAN', 'MGW']
Saved: ['SCE', 'STT', 'MHK']
Saved: ['OKC', 'RIC', 'IDA']
Saved: ['BUR', 'SJT', 'SAV']
Saved: ['LAS', 'SGF', 'CHS']
Saved: ['ITO', 'GNV', 'FOD']
Saved: ['PWM', 'MQT', 'GRR']
Saved: ['MIA', 'SCC', 'BIS']
Saved: ['ECP', 'AVP', 'DLH']
Saved: ['PSP', 'STC', 'ORF']
Saved: ['CLL', 'HRL', 'VPS']
Saved: ['DIK', 'FAI', 'BOS']
Saved: ['XWA', 'MEI', 'MDT']
Saved: ['BRD', 'SBP', 'PPG']
Saved: ['AKN', 'HLN', 'WRG']
Saved: ['EAR', 'LEX', 'BUF']
Saved: ['OTH', 'MHT', 'CMX']
Saved: ['TUL', 'BIH', 'MLB']
Saved: ['GSP', 'MCW', 'LBL']
Saved: ['CPR', 'GEG', 'GUC']
Saved: ['LAR', 'SLC', 'DRO']
Saved: ['PIT', 'TYR', 'BQN']
Saved: ['SUN', 'BZN', 'OME']
Saved: ['SRQ', 'DEN', 'BLI']
Saved: ['SFB', 'CRP', 'LGB']
Saved: ['ILM', 'DAB', 'SJC']
Saved: ['MSY', 'SPI', 'CLE']
Saved: ['JNU',

In [42]:
weather_dir = Path("../data/raw/weather")

weather_files = list(weather_dir.glob("*.parquet"))

weather = pd.concat(
    [pd.read_parquet(file) for file in weather_files],
    ignore_index=True
)

In [43]:
weather.shape

(3074760, 10)

In [68]:
print(f"There should be 350 airports * 365 days * 24 hours rows in the data: {350*365*24}")

There should be 350 airports * 365 days * 24 hours rows in the data: 3066000


In [69]:
# the difference comes because we looped through the old airport csv by mistake which had 351 airports due to appending the PBI airport 
# and not removing the renamed version

In [44]:
weather.groupby("iata_code").size().describe()

count     351.0
mean     8760.0
std         0.0
min      8760.0
25%      8760.0
50%      8760.0
75%      8760.0
max      8760.0
dtype: float64

In [45]:
weather.groupby("iata_code").size().loc[lambda x: x != 8760]

Series([], dtype: int64)

In [46]:
weather.duplicated(
    subset=["iata_code", "time"]
).sum()

np.int64(0)

In [47]:
weather.isna().mean().sort_values(ascending=False)

time                0.0
temperature_2m      0.0
precipitation       0.0
snowfall            0.0
snow_depth          0.0
visibility          0.0
surface_pressure    0.0
wind_speed_10m      0.0
wind_gusts_10m      0.0
iata_code           0.0
dtype: float64

In [70]:
# all airports have the right number of rows and no missing data

In [48]:
weather = weather.rename(
    columns={"time": "WEATHER_HOUR"}
) #will help with the merge later

In [49]:
weather.to_parquet(
    "../data/processed/weather_2025.parquet",
    index=False
) # save this to use in the pipeline 

In [53]:
dep_weather = weather.rename(
    columns={
        "iata_code": "ORIGIN",
        "WEATHER_HOUR": "DEP_WEATHER_HOUR",
        "temperature_2m": "DEP_TEMPERATURE",
        "precipitation": "DEP_PRECIPITATION",
        "snowfall": "DEP_SNOWFALL",
        "snow_depth": "DEP_SNOW_DEPTH",
        "visibility": "DEP_VISIBILITY",
        "surface_pressure": "DEP_SURFACE_PRESSURE",
        "wind_speed_10m": "DEP_WIND_SPEED",
        "wind_gusts_10m": "DEP_WIND_GUSTS",
    }
) 

In [73]:
df = df.merge(
    dep_weather,
    on=["ORIGIN", "DEP_WEATHER_HOUR"],
    how="left",
    validate="many_to_one"
)

In [ ]:
# create two versions of weather: dep_weather and arr_weather because we will merge twice

In [54]:
dep_weather.head(2)

,DEP_WEATHER_HOUR,DEP_TEMPERATURE,DEP_PRECIPITATION,DEP_SNOWFALL,DEP_SNOW_DEPTH,DEP_VISIBILITY,DEP_SURFACE_PRESSURE,DEP_WIND_SPEED,DEP_WIND_GUSTS,ORIGIN
0,2025-01-01 00:00:00+00:00,17.3,0.0,0.0,0.0,25400.0,1002.1,10.2,28.1,AGS
1,2025-01-01 01:00:00+00:00,17.3,0.0,0.0,0.0,28100.0,1002.8,8.8,21.6,AGS


In [56]:
df.head(2)

,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,ORIGIN,DEST,CRS_DEP_TIME,CRS_ARR_TIME,CRS_ELAPSED_TIME,DISTANCE,...,ORIGIN_LONGITUDE,ORIGIN_TIMEZONE,DEST_LATITUDE,DEST_LONGITUDE,DEST_TIMEZONE,SCHEDULED_DEP_DATETIME,SCHEDULED_DEP_UTC,SCHEDULED_ARR_UTC,DEP_WEATHER_HOUR,ARR_WEATHER_HOUR
0,1,1,1,3,AUS,ORD,545,830,165.0,977.0,...,-97.662015,America/Chicago,41.978600,-87.904800,America/Chicago,2025-01-01 05:45:00,2025-01-01 11:45:00+00:00,2025-01-01 14:30:00+00:00,2025-01-01 11:00:00+00:00,2025-01-01 14:00:00+00:00
1,1,1,7,2,PBI,DFW,620,842,202.0,1102.0,...,-80.095600,America/New_York,32.896801,-97.038002,America/Chicago,2025-01-07 06:20:00,2025-01-07 11:20:00+00:00,2025-01-07 14:42:00+00:00,2025-01-07 11:00:00+00:00,2025-01-07 14:00:00+00:00


In [60]:
df.shape

(360000, 24)

In [63]:
test = df.merge(dep_weather,on=["ORIGIN", "DEP_WEATHER_HOUR"], how="left")
test[test['DEP_TEMPERATURE'].isna()]['DAY_OF_MONTH'].value_counts() 

DAY_OF_MONTH
31    171
Name: count, dtype: int64

In [ ]:
# looks like the final day of weather was not included
# drop in the pipeline

In [64]:
arr_weather = weather.rename(
    columns={
        "iata_code": "DEST",
        "WEATHER_HOUR": "ARR_WEATHER_HOUR",
        "temperature_2m": "ARR_TEMPERATURE",
        "precipitation": "ARR_PRECIPITATION",
        "snowfall": "ARR_SNOWFALL",
        "snow_depth": "ARR_SNOW_DEPTH",
        "visibility": "ARR_VISIBILITY",
        "surface_pressure": "ARR_SURFACE_PRESSURE",
        "wind_speed_10m": "ARR_WIND_SPEED",
        "wind_gusts_10m": "ARR_WIND_GUSTS",
    }
)

df = df.merge(
    arr_weather,
    on=["DEST", "ARR_WEATHER_HOUR"],
    how="left",
    validate="many_to_one"
)

In [81]:
df[df["ARR_TEMPERATURE"].isna()]["DAY_OF_MONTH"].value_counts()

DAY_OF_MONTH
31    294
Name: count, dtype: int64

In [82]:
df["DAY_OF_MONTH"].value_counts()

DAY_OF_MONTH
13    12163
23    12162
7     12102
21    12089
2     12075
24    12054
14    12050
20    12032
26    12012
28    11997
10    11914
17    11901
18    11892
22    11882
6     11868
3     11829
9     11803
11    11795
27    11743
15    11736
16    11714
19    11685
12    11663
5     11644
1     11626
25    11612
8     11600
4     11427
29    10918
30    10736
31     6276
Name: count, dtype: int64

In [77]:
df.isna().sum() #all data 

QUARTER                     0
MONTH                       0
DAY_OF_MONTH                0
DAY_OF_WEEK                 0
ORIGIN                      0
DEST                        0
CRS_DEP_TIME                0
CRS_ARR_TIME                0
CRS_ELAPSED_TIME            0
DISTANCE                    0
AIRLINE                     0
ROUTE                       0
DELAY                       0
ORIGIN_LATITUDE             0
ORIGIN_LONGITUDE            0
ORIGIN_TIMEZONE             0
DEST_LATITUDE               0
DEST_LONGITUDE              0
DEST_TIMEZONE               0
SCHEDULED_DEP_DATETIME      0
SCHEDULED_DEP_UTC           0
SCHEDULED_ARR_UTC           0
DEP_WEATHER_HOUR            0
ARR_WEATHER_HOUR            0
ARR_TEMPERATURE           294
ARR_PRECIPITATION         294
ARR_SNOWFALL              294
ARR_SNOW_DEPTH            294
ARR_VISIBILITY            294
ARR_SURFACE_PRESSURE      294
ARR_WIND_SPEED            294
ARR_WIND_GUSTS            294
DEP_TEMPERATURE           171
DEP_PRECIP

In [ ]:
#TODO JOIN THIS WEATHER DATA TWICE ON THE DATAFRAME

In [38]:
test_batch = airports.iloc[:3]

test_weather = get_weather_batch(test_batch)

test_weather.shape

(26280, 10)

In [39]:
test_weather.head()

,time,temperature_2m,precipitation,snowfall,snow_depth,visibility,surface_pressure,wind_speed_10m,wind_gusts_10m,iata_code
0,2025-01-01T00:00,5.5,0.0,0.0,0.0,20700.0,1000.1,23.8,54.4,EVV
1,2025-01-01T01:00,5.1,0.0,0.0,0.0,18800.0,1001.3,25.8,52.2,EVV
2,2025-01-01T02:00,4.9,0.0,0.0,0.0,18700.0,1001.9,23.9,54.0,EVV
3,2025-01-01T03:00,4.6,0.0,0.0,0.0,18400.0,1002.7,23.4,42.8,EVV
4,2025-01-01T04:00,4.4,0.0,0.0,0.0,19900.0,1003.4,24.4,45.7,EVV


In [ ]:
# will join on time and iata_code for DEST and ARR 

In [40]:
test_weather.groupby('iata_code').size()

iata_code
CDV    8760
EVV    8760
HDN    8760
dtype: int64

## Pipeline Steps

1. Create a list of all the airline codes
2. Read in the airport coordinates (with the right columns)
3. Merge the codes and coordinates
4. Add the timezone
5. Merge on ORIGIN codes
6. Merge on DEPT codes
7. Create SCHEDULED DEPT and ARR datetime columns
8. Convert these to UTC (maybe optional if you can use the timezone in the weather API
9. Floor/round to get in date hour format e.g. 2025-01-01 14:00:00+00:00
10. Load in weather data
11. Create a dep_weather and arrival_weather
12. Join each onto main dataframe
13. Drop unnecessary columns
14. Done!